<a href="https://colab.research.google.com/github/pedrosouzag/edicao-imagens-ia/blob/main/semana2ynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
from google.colab.patches import cv2_imshow
uploaded = files.upload()

In [ ]:
import cv2 as cv
import numpy as np
from google.colab.patches import cv2_imshow

In [ ]:

def rescaleFrame(frame, scale = 0.75):
  width = int(frame.shape[1] * scale)
  height = int(frame.shape[0] * scale)
  dimensions = (width, height)

  return cv.resize(frame, dimensions, interpolation=cv.INTER_AREA)


In [ ]:

nome_imagem = list(uploaded.keys())[0]

img = cv.imread(nome_imagem)
resized_img = rescaleFrame(img)

print("original\n")
cv2_imshow(img)

print("redimensionada\n")
cv2_imshow(resized_img)

In [ ]:
# Converting to Gray Scale
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
cv2_imshow(gray)

# Converte cinza para preto e branco
ret, preto_branco = cv.threshold(gray, 127, 255, cv.THRESH_BINARY)

# Mostra
cv2_imshow(preto_branco)

In [ ]:
# Blur image

blur = cv.GaussianBlur(img, (3,3), cv.BORDER_DEFAULT)
cv2_imshow(blur)

blur2 = cv.GaussianBlur(img, (9,9), cv.BORDER_DEFAULT)
cv2_imshow(blur2)


In [ ]:
# Edge Cascade

canny = cv.Canny(img, 125, 175)
cv2_imshow(canny)

canny = cv.Canny (blur, 125,175)
cv2_imshow(canny)

canny = cv.Canny(blur2, 125, 175)
cv2_imshow(canny)

In [ ]:
# Dilate

dilated = cv.dilate(canny, (7,7), iterations=3)
cv2_imshow(dilated)


In [ ]:
# Translation
def translate(img, x, y):
  transMat = np.float32([[1,0,x], [0,1,y]])
  dimensions = (img.shape[1], img.shape[0])
  return cv.warpAffine(img, transMat, dimensions)


# -x esquerda
# -y cima
# x direita
# y baixo

translated = translate (img, 100, 100)
cv2_imshow(translated)


In [ ]:
# Rotation

def rotate(img, angle, rotPoint = None):
  # o img shape retorna altura, largura e canais, usando esse :2, pega so os dois primeiros
  (height, width) = img.shape[:2]

  if rotPoint is None:
    #inverter altura e largura devido ao OPENCV
    rotPoint = (width//2, height//2)

    rotMat = cv.getRotationMatrix2D(rotPoint, angle, scale = 1.0)
    dimensions = (width, height)

    return cv.warpAffine(img, rotMat, dimensions)

rotated = rotate(img, 45)
cv2_imshow(rotated)

rotated_rotated = rotate (rotated, -45)
cv2_imshow(rotated_rotated)




In [ ]:
# Resize

#Cubic melhor para aumentar, AREA melhor para reduzir
resized = cv.resize(img, (100, 100), interpolation = cv.INTER_CUBIC)
cv2_imshow (resized)



In [ ]:
#Fliping

flip = cv.flip(img, 0)
cv2_imshow(flip)

In [ ]:
blank = np.zeros((400,400, 3), dtype= 'uint8')

rectangle = cv.rectangle(blank.copy(), (30,30), (370,370), (255,195,255), -1)
circle = cv.circle(blank.copy(), (200,200), 200, (255,0,100), -1)

#cv2_imshow(rectangle)
#cv2_imshow(circle)

bitwise_and = cv.bitwise_and(rectangle, circle)
cv2_imshow(bitwise_and)

# tem como usar com or, xor, etc





In [ ]:
# MASK

blank = np.zeros(img.shape[:2], dtype='uint8')

mask = cv.circle(blank.copy(), (img.shape[1]//2, img.shape[0]//2), 200, 255, -1)
cv2_imshow(mask)

masked = cv.bitwise_and(img, img, mask=mask)
cv2_imshow(masked)

mask = cv.circle(img.copy(), (img.shape[1]//2, img.shape[0]//2), 200, 255, -1)
cv2_imshow(mask)

mask = cv.circle(blank.copy(), (img.shape[1]//3, img.shape[0]//4), 200, 255, -1)
cv2_imshow(mask)

masked = cv.bitwise_and(img, img, mask=mask)
cv2_imshow(masked)




In [ ]:
# Conversao para HSV
hsv = cv.cvtColor(img, cv.COLOR_BGR2HSV)
cv2_imshow(hsv)

# Conversao para LAB (L = luminosidade, A e B = cores)
lab = cv.cvtColor(img, cv.COLOR_BGR2LAB)
cv2_imshow(lab)

# Separando os canais do HSV
h, s, v = cv.split(hsv)
print("Canal H (matiz - a cor em si)")
cv2_imshow(h)

print("Canal S (saturacao - intensidade da cor)")
cv2_imshow(s)

print("Canal V (value - brilho)")
cv2_imshow(v)

# Separando os canais do BGR original
b, g, r = cv.split(img)
print("Blue")
cv2_imshow(b)

print("Green")
cv2_imshow(g)

print("Red")
cv2_imshow(r)

In [ ]:
# convertScaleAbs faz: novo_pixel = alpha * pixel + beta
# alpha controla CONTRASTE (1.0 = mantem igual, >1 aumenta, <1 diminui)
# beta controla BRILHO (soma/subtrai de cada pixel)

# Clarear (aumenta brilho)
clareada = cv.convertScaleAbs(img, alpha=1.0, beta=50)
cv2_imshow(clareada)

# Escurecer (diminui brilho)
escurecida = cv.convertScaleAbs(img, alpha=1.0, beta=-50)
cv2_imshow(escurecida)

# Aumentar contraste
mais_contraste = cv.convertScaleAbs(img, alpha=1.5, beta=0)
cv2_imshow(mais_contraste)

# Diminuir contraste
menos_contraste = cv.convertScaleAbs(img, alpha=0.5, beta=0)
cv2_imshow(menos_contraste)

In [ ]:
hsv_float = cv.cvtColor(img, cv.COLOR_BGR2HSV).astype('float32')

# Aumenta saturacao em 50% (canal S = indice 1)
hsv_float[:, :, 1] = hsv_float[:, :, 1] * 1.5
hsv_float[:, :, 1] = np.clip(hsv_float[:, :, 1], 0, 255)

img_saturada = cv.cvtColor(hsv_float.astype('uint8'), cv.COLOR_HSV2BGR)
cv2_imshow(img_saturada)

# Diminui saturacao (deixa mais proximo de cinza)
hsv_float2 = cv.cvtColor(img, cv.COLOR_BGR2HSV).astype('float32')
hsv_float2[:, :, 1] = hsv_float2[:, :, 1] * 0.3
img_dessaturada = cv.cvtColor(hsv_float2.astype('uint8'), cv.COLOR_HSV2BGR)
cv2_imshow(img_dessaturada)


In [ ]:
blank = np.zeros(img.shape[:2], dtype='uint8')

# mascara circular no centro da imagem
mask_regiao = cv.circle(blank.copy(), (img.shape[1]//2, img.shape[0]//2), 150, 255, -1)
cv2_imshow(mask_regiao)

# escurece a imagem inteira, mas so vamos usar o pedaco de dentro do circulo
img_toda_escurecida = cv.convertScaleAbs(img, alpha=1.0, beta=-80)

# copia da imagem original pra editar
img_regiao_editada = img.copy()

# onde a mascara e 255 (dentro do circulo), troca pelo pixel escurecido
img_regiao_editada[mask_regiao == 255] = img_toda_escurecida[mask_regiao == 255]

cv2_imshow(img_regiao_editada)

In [ ]:
# borrar as mascara
mask_suave = cv.GaussianBlur(mask_regiao, (51, 51), 0)
cv2_imshow(mask_suave)



In [ ]:
# Converte a máscara de 0–255 para 0.0–1.0
mask_norm = mask_suave.astype('float32') / 255

# Repete a máscara nos canais B, G e R
mask_norm_3canais = cv.merge([mask_norm, mask_norm, mask_norm])

# Converte as imagens para float para fazer os cálculos
editada_float = img_toda_escurecida.astype('float32')
original_float = img.astype('float32')

# Mistura a imagem escura com a original usando a máscara
blended = (editada_float * mask_norm_3canais + original_float * (1 - mask_norm_3canais))
#blended = original_float * (1 - mask_norm_3canais)
#blended = editada_float * (1 - mask_norm_3canais)

# Volta para o formato normal de imagem
blended = blended.astype('uint8')

cv2_imshow(blended)